# EDA Mejorado: Dataset Sintético v2 con Defectos Expandidos
## Análisis de Datos Sintéticos + Validaciones de Calidad

Este notebook carga el dataset generado y aplica análisis de calidad similares a los del pipeline original (Auditoria3):
- Validación de dominios
- Cruce flota-telemetría
- Cobertura de dispositivos
- Análisis de defectos inyectados
- Detección de anomalías

## 1. Montar Google Drive y Cargar Dataset

In [ ]:
from google.colab import drive
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
from datetime import datetime

drive.mount('/content/drive')
print("✓ Google Drive montado")

In [ ]:
# Cargar dataset desde Google Drive
dataset_dir = Path('/content/drive/MyDrive/Integrador/datasets/early_stage_v2')

# Cargar todos los CSVs
tables = {}
for csv_file in dataset_dir.glob('*.csv'):
    table_name = csv_file.stem
    tables[table_name] = pd.read_csv(csv_file)
    print(f"✓ {table_name:30s} {len(tables[table_name]):6d} filas")

# Cargar manifest
manifest_path = dataset_dir / 'manifest.json'
if manifest_path.exists():
    with open(manifest_path) as f:
        manifest = json.load(f)
    print(f"\n✓ Manifest cargado (seed: {manifest['seed']})")

print(f"\n📊 Total de datos cargados: {sum(len(df) for df in tables.values()):,} registros")

## 2. Ground Truth: Defectos Inyectados

In [ ]:
# Cargar tabla de defectos
gt = tables.get('ground_truth', pd.DataFrame())

if len(gt) > 0:
    print(f"🔴 ANÁLISIS DE DEFECTOS INYECTADOS")
    print(f"\nTotal de defectos: {len(gt)}")
    
    # Por tipo
    print(f"\nDistribución por tipo:")
    tipo_counts = gt['tipo'].value_counts().sort_values(ascending=False)
    for tipo, count in tipo_counts.items():
        print(f"  {tipo:30s} {count:3d}")
    
    # Por severidad
    print(f"\nDistribución por severidad:")
    sev_counts = gt['severidad'].value_counts()
    for sev, count in sev_counts.items():
        print(f"  {sev:30s} {count:3d}")
    
    # Por entidad
    print(f"\nDistribución por entidad:")
    ent_counts = gt['entidad'].value_counts()
    for ent, count in ent_counts.items():
        print(f"  {ent:30s} {count:3d}")
else:
    print("⚠ No hay tabla ground_truth")

In [ ]:
# Visualización de defectos por tipo
if len(gt) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Tipo
    tipo_counts.plot(kind='barh', ax=axes[0], color='#FF6B6B')
    axes[0].set_title('Defectos por Tipo', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Cantidad')
    
    # Severidad
    colors = {'alta': '#FF6B6B', 'media': '#FFA500', 'baja': '#FFD700'}
    sev_counts.plot(kind='bar', ax=axes[1], color=[colors.get(s, '#999') for s in sev_counts.index])
    axes[1].set_title('Defectos por Severidad', fontsize=12, fontweight='bold')
    axes[1].set_ylabel('Cantidad')
    axes[1].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()

## 3. Validación de Dominios

In [ ]:
# Análisis de dominios en tabla vehiculo
vehiculos = tables.get('vehiculo', pd.DataFrame())

if len(vehiculos) > 0:
    print("🔍 VALIDACIÓN DE DOMINIOS")
    print(f"\nTotal vehículos: {len(vehiculos)}")
    
    # Detectar dominios duplicados
    dominios_unicos = vehiculos['dominio_sintetico'].nunique()
    dominios_duplicados = len(vehiculos) - dominios_unicos
    
    print(f"\nDominios únicos: {dominios_unicos}")
    print(f"Dominios duplicados (registros): {dominios_duplicados}")
    
    # Top dominios duplicados
    dup_count = vehiculos['dominio_sintetico'].value_counts()
    dup_count = dup_count[dup_count > 1]
    
    if len(dup_count) > 0:
        print(f"\nTop 10 dominios con duplicados:")
        for dom, cnt in dup_count.head(10).items():
            print(f"  {dom:20s} {cnt:3d} vehículos")
    
    # Formatos inconsistentes
    print(f"\n📊 Análisis de formato de dominios:")
    print(f"  Largo promedio: {vehiculos['dominio_sintetico'].str.len().mean():.1f}")
    print(f"  Largo mín/máx: {vehiculos['dominio_sintetico'].str.len().min()}/{vehiculos['dominio_sintetico'].str.len().max()}")

## 4. Cruce Flota-Telemetría

In [ ]:
# Análisis de cobertura: qué vehículos tienen dispositivos
dispositivos = tables.get('dispositivo', pd.DataFrame())
telemetria = tables.get('evento_telemetria', pd.DataFrame())

if len(vehiculos) > 0 and len(dispositivos) > 0:
    print("🔗 CRUCE FLOTA-TELEMETRÍA")
    print(f"\nVehículos totales: {len(vehiculos)}")
    print(f"Dispositivos registrados: {len(dispositivos)}")
    print(f"Eventos de telemetría: {len(telemetria)}")
    
    # Vehículos con dispositivo
    veh_con_disp = vehiculos['id'].isin(dispositivos['vehiculo_id']).sum()
    veh_sin_disp = len(vehiculos) - veh_con_disp
    
    print(f"\nCobertura de dispositivos:")
    print(f"  Con dispositivo: {veh_con_disp} ({100*veh_con_disp/len(vehiculos):.1f}%)")
    print(f"  Sin dispositivo: {veh_sin_disp} ({100*veh_sin_disp/len(vehiculos):.1f}%)")
    
    # Dispositivos por estado de transmisión
    print(f"\nDispositivos por estado:")
    for estado, count in dispositivos['estado_transmision'].value_counts().items():
        print(f"  {estado:20s} {count:3d}")
    
    # Vehículos con eventos telemetría
    veh_con_tel = telemetria['dispositivo_id'].nunique() if len(telemetria) > 0 else 0
    print(f"\nDisositivos con eventos: {veh_con_tel}")

In [ ]:
# Visualizar cobertura
if len(vehiculos) > 0 and len(dispositivos) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # Pie chart de cobertura
    cobertura_data = [veh_con_disp, veh_sin_disp]
    axes[0].pie(cobertura_data, labels=['Con dispositivo', 'Sin dispositivo'], autopct='%1.1f%%',
                colors=['#4CAF50', '#FF9800'])
    axes[0].set_title('Cobertura de Dispositivos', fontsize=12, fontweight='bold')
    
    # Estado de transmisión
    estado_counts = dispositivos['estado_transmision'].value_counts()
    estado_counts.plot(kind='bar', ax=axes[1], color=['#4CAF50', '#FFC107', '#F44336'])
    axes[1].set_title('Dispositivos por Estado de Transmisión', fontsize=12, fontweight='bold')
    axes[1].set_ylabel('Cantidad')
    axes[1].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()

## 5. Análisis de Transacciones de Combustible

In [ ]:
transacciones = tables.get('transaccion_combustible', pd.DataFrame())

if len(transacciones) > 0:
    print("⛽ ANÁLISIS DE COMBUSTIBLE")
    print(f"\nTransacciones totales: {len(transacciones)}")
    
    # Valores faltantes
    print(f"\nValores faltantes:")
    for col in ['litros', 'precio_unitario', 'importe_total']:
        if col in transacciones.columns:
            missing = transacciones[col].isna().sum()
            print(f"  {col:20s} {missing:5d} ({100*missing/len(transacciones):.1f}%)")
    
    # Valores inválidos (no numéricos)
    print(f"\nValores no numéricos en 'litros':")
    invalid_litros = transacciones['litros'].apply(lambda x: isinstance(x, str)).sum()
    print(f"  Detectados: {invalid_litros}")
    
    # Estadísticas de litros
    litros_numeric = pd.to_numeric(transacciones['litros'], errors='coerce')
    print(f"\nEstadísticas de litros (válidos):")
    print(f"  Media: {litros_numeric.mean():.2f} L")
    print(f"  Mediana: {litros_numeric.median():.2f} L")
    print(f"  Desv. Est.: {litros_numeric.std():.2f} L")
    print(f"  Rango: [{litros_numeric.min():.2f}, {litros_numeric.max():.2f}] L")

## 6. Detección de Anomalías

In [ ]:
# Detección de anomalías basada en ground_truth
if len(gt) > 0:
    print("⚠️  ANOMALÍAS DETECTADAS POR TIPO")
    
    # DQ_DUP_VEH_ID: duplicados de matrícula
    dup_veh_records = gt[gt['tipo'] == 'DQ_DUP_VEH_ID']
    if len(dup_veh_records) > 0:
        dup_veh_ids = dup_veh_records['registro_id'].unique()
        affected_veh = vehiculos[vehiculos['id'].isin(dup_veh_ids)]
        print(f"\n🔴 DQ_DUP_VEH_ID (Matrículas duplicadas)")
        print(f"   Registros afectados: {len(dup_veh_ids)}")
        print(f"   Ejemplo: {affected_veh['matricula_sintetica'].iloc[0] if len(affected_veh) > 0 else 'N/A'}")
    
    # DQ_DUP_DOMAIN: duplicados de dominio
    dup_dom_records = gt[gt['tipo'] == 'DQ_DUP_DOMAIN']
    if len(dup_dom_records) > 0:
        dup_dom_ids = dup_dom_records['registro_id'].unique()
        affected_dom = vehiculos[vehiculos['id'].isin(dup_dom_ids)]
        print(f"\n🔴 DQ_DUP_DOMAIN (Dominios duplicados)")
        print(f"   Registros afectados: {len(dup_dom_ids)}")
        print(f"   Ejemplo: {affected_dom['dominio_sintetico'].iloc[0] if len(affected_dom) > 0 else 'N/A'}")
    
    # DQ_FORMAT_DRIFT
    format_records = gt[gt['tipo'] == 'DQ_FORMAT_DRIFT']
    if len(format_records) > 0:
        print(f"\n🟡 DQ_FORMAT_DRIFT (Formato inconsistente)")
        print(f"   Registros afectados: {len(format_records)}")
        # Desglose por campo
        for param_str in format_records['parametros'].unique()[:3]:
            try:
                param = json.loads(param_str)
                print(f"   - {param.get('campo', 'N/A')}: {param.get('problema', 'N/A')}")
            except:
                pass
    
    # DQ_MISSING_VALUE
    missing_records = gt[gt['tipo'] == 'DQ_MISSING_VALUE']
    if len(missing_records) > 0:
        print(f"\n🟡 DQ_MISSING_VALUE (Valores faltantes)")
        print(f"   Registros afectados: {len(missing_records)}")
    
    # DQ_INVALID_TYPE
    invalid_records = gt[gt['tipo'] == 'DQ_INVALID_TYPE']
    if len(invalid_records) > 0:
        print(f"\n🟡 DQ_INVALID_TYPE (Tipo de dato inválido)")
        print(f"   Registros afectados: {len(invalid_records)}")
    
    # DQ_INCONSISTENCY
    inconsist_records = gt[gt['tipo'] == 'DQ_INCONSISTENCY']
    if len(inconsist_records) > 0:
        print(f"\n🔴 DQ_INCONSISTENCY (Valores inconsistentes)")
        print(f"   Registros afectados: {len(inconsist_records)}")
        # Por problema
        for param_str in inconsist_records['parametros'].unique()[:3]:
            try:
                param = json.loads(param_str)
                print(f"   - {param.get('problema', 'N/A')}")
            except:
                pass

## 7. Resumen por Subunidad

In [ ]:
# Resumen de defectos por tabla/entidad
if len(gt) > 0:
    print("📊 RESUMEN DE DEFECTOS POR ENTIDAD")
    
    defectos_por_entidad = gt['entidad'].value_counts()
    for entidad, count in defectos_por_entidad.items():
        print(f"\n{entidad.upper()}:")
        tipos = gt[gt['entidad'] == entidad]['tipo'].value_counts()
        for tipo, tipo_count in tipos.items():
            print(f"  {tipo:30s} {tipo_count:3d}")

## 8. Matriz de Validación Final

In [ ]:
# Resumen ejecutivo
print("\n" + "="*70)
print("📋 RESUMEN EJECUTIVO DEL DATASET")
print("="*70)

print(f"\n📊 VOLUMEN:")
for table_name, df in sorted(tables.items()):
    if table_name != 'ground_truth' and table_name != 'ejecucion_dataset':
        print(f"  {table_name:30s} {len(df):8d} registros")

print(f"\n🔴 DEFECTOS:")
print(f"  Total inyectados: {len(gt)}")
print(f"  Alta severidad: {(gt['severidad'] == 'alta').sum()}")
print(f"  Media severidad: {(gt['severidad'] == 'media').sum()}")
print(f"  Baja severidad: {(gt['severidad'] == 'baja').sum()}")

print(f"\n✅ ESTADO:")
print(f"  Dataset generado: {manifest.get('generated_at', 'N/A')}")
print(f"  Seed reproducible: {manifest.get('seed')}")
print(f"  Escenario: {manifest.get('scenario')}")

print("\n" + "="*70)